[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maxischa/datacamp_test/blob/main/bloc3_stats/corrections/seance2_correction.ipynb)

# Séance 3.2 — Comparer deux groupes — hasard ou vrai écart ?

**Correction** · durée : 2h (≈50 min de cours, ≈50 min d'exercices)

> ⚠️ **Avant de taper quoi que ce soit :** *Fichier → Enregistrer une copie dans Drive*. Sinon votre travail sera perdu en fermant l'onglet.
>
> 📱 Sur tablette, faites d'abord les réglages de [Bien démarrer](https://github.com/maxischa/datacamp_test/blob/main/ressources/setup_tablette.md).

## Objectifs

À la fin de cette séance, vous saurez :

- expliquer pourquoi une moyenne calculée sur un échantillon n'est jamais exacte
- construire un intervalle de confiance à 95 % par rééchantillonnage
- comparer deux groupes avec un test t et lire sa p-value
- distinguer « pas de différence » de « pas de différence détectable »
- repérer les deux pièges qui rendent un test faux : dépendance et tests répétés

## Correction

Solutions commentées. Comparez avec ce que vous aviez écrit : plusieurs formulations peuvent être correctes.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

# Affichage adapte aux petits ecrans
pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 80)

# Les donnees sont lues directement depuis le web : rien a telecharger
BASE = "https://raw.githubusercontent.com/maxischa/datacamp_test/main/bloc3_stats/data/"

In [ ]:
def verifier(nom, condition, indice=""):
    """Affiche un retour immediat sans interrompre le notebook."""
    print("OK   -", nom) if condition else print("A REVOIR -", nom, ":", indice)

Chargement des données utilisées dans toute la feuille :

In [ ]:
cmd = pd.read_csv(BASE + "commandes.csv")

fr = cmd.query("pays == 'France'")["ca"]
de = cmd.query("pays == 'Allemagne'")["ca"]
print(len(fr), "commandes francaises,", len(de), "allemandes")

### Exercice 1 — Deux marchés à comparer

> **Votre mission :**
> - Extraire les paniers belges dans `be` et espagnols dans `es`.
> - Afficher les effectifs et les moyennes ; mettre la moyenne belge arrondie à 2 décimales dans `moy_be`.

In [ ]:
be = cmd.query("pays == 'Belgique'")["ca"]
es = cmd.query("pays == 'Espagne'")["ca"]

moy_be = round(be.mean(), 2)
print(len(be), "commandes belges, moyenne", moy_be)
print(len(es), "commandes espagnoles, moyenne", round(es.mean(), 2))

In [ ]:
verifier("1 - moyenne belge", moy_be == 446.7,
         "guillemets doubles a l'exterieur, simples autour du nom du pays")

### Exercice 2 — L'intervalle de confiance belge

> **Votre mission :**
> - Rééchantillonner `be` 1000 fois (avec remise, `random_state=i`) → `boot_be`.
> - En tirer les bornes de l'intervalle à 95 % → `bas_be` et `haut_be` (arrondies à 2 décimales).

In [ ]:
# replace=True : tirage avec remise, sinon on retire toujours
# exactement les memes commandes et la moyenne ne bouge jamais
boot_be = pd.Series([be.sample(len(be), replace=True, random_state=i).mean()
                     for i in range(1000)])

# 95 % centraux : on coupe 2,5 % en bas et 2,5 % en haut
bas_be = round(boot_be.quantile(0.025), 2)
haut_be = round(boot_be.quantile(0.975), 2)
print("panier belge : entre", bas_be, "et", haut_be)

In [ ]:
verifier("2a - borne basse", bas_be == 384.58, "quantile(0.025)")
verifier("2b - borne haute", haut_be == 521.95,
         "quantile(0.975), et replace=True dans le sample")

### Exercice 3 — Belgique contre Espagne

> **Votre mission :**
> - Tester l'écart entre `be` et `es` → `p_be_es` (arrondie à 4 décimales).
> - Conclut-on à un écart au seuil de 5 % ? Mettre `True` ou `False` dans `conclut`.

In [ ]:
p_be_es = round(stats.ttest_ind(be, es, equal_var=False).pvalue, 4)

conclut = p_be_es < 0.05
print("p =", p_be_es, "| on conclut a un ecart :", conclut)

# 0,068 : juste au-dessus du seuil. L'ecart de 157 EUR est important,
# mais 72 et 64 commandes ne suffisent pas a l'etablir. Ce n'est pas
# "pas de difference" : c'est "pas assez de donnees pour le dire".

In [ ]:
verifier("3a - p-value Belgique/Espagne", p_be_es == 0.0683,
         "stats.ttest_ind(be, es, equal_var=False).pvalue")
verifier("3b - conclusion au seuil de 5 %", not conclut,
         "0,0683 est-il inferieur a 0,05 ?")

### Exercice 4 — Le jour de la semaine compte-t-il ?

> **Votre mission :**
> - Comparer les paniers du **jeudi** (jour le plus chargé) et du **dimanche** → `p_jour` (arrondie à 4 décimales).
> - Le jour de la semaine influence-t-il le montant des commandes ?

In [ ]:
je = cmd.query("jour == 'jeudi'")["ca"]
di = cmd.query("jour == 'dimanche'")["ca"]

p_jour = round(stats.ttest_ind(je, di, equal_var=False).pvalue, 4)
print("p =", p_jour)

# p = 0,2975 : rien ne permet de dire que le jour change le panier.

In [ ]:
verifier("4 - p-value jeudi/dimanche", p_jour == 0.2975,
         "le resultat du test a un attribut .pvalue")

### Exercice 5 — Changer de variable, pas de méthode

> **Votre mission :**
> - Même comparaison France / Allemagne, mais sur le nombre d'**articles** (`qte`) → `p_qte`.
> - Attention : `fr` et `de` contiennent le `ca`. Il faut repartir de `cmd`.

In [ ]:
qfr = cmd.query("pays == 'France'")["qte"]
qde = cmd.query("pays == 'Allemagne'")["qte"]

p_qte = round(stats.ttest_ind(qfr, qde, equal_var=False).pvalue, 4)
print(round(qfr.mean(), 2), "contre", round(qde.mean(), 2), "articles | p =", p_qte)

# Meme conclusion que sur les euros : rien a departager.

In [ ]:
verifier("5 - p-value sur les quantites", p_qte == 0.7751,
         "la colonne des articles s'appelle qte")

### Exercice 6 — Le prix d'un petit effectif

> **Votre mission :**
> - Le Japon affiche le panier moyen le plus élevé du fichier. Sur combien de commandes ?
> - Calculer la **largeur** de son intervalle de confiance à 95 % → `larg_jp` (arrondie à 2 décimales).
> - Comparez-la à la largeur française (184,29 €).

In [ ]:
jp = cmd.query("pays == 'Japon'")["ca"]
bj = pd.Series([jp.sample(len(jp), replace=True, random_state=i).mean()
                for i in range(1000)])

larg_jp = round(bj.quantile(0.975) - bj.quantile(0.025), 2)
print(len(jp), "commandes | largeur de l'intervalle :", larg_jp, "euros")

# 2 193 EUR de largeur contre 184 pour la France : sur 12 commandes,
# la moyenne n'est plus une mesure, c'est une impression.

In [ ]:
verifier("6 - largeur de l'intervalle japonais", larg_jp == 2193.18,
         "quantile(0.975) moins quantile(0.025)")

### Exercice 7 — Compter les individus, pas les lignes

> **Votre mission :**
> - Combien de clients distincts derrière les commandes irlandaises ? → `cli_irl`
> - Et derrière les britanniques ? → `cli_uk`

In [ ]:
# nunique compte les valeurs DISTINCTES, count compterait les lignes
cli_irl = cmd.query("pays == 'Irlande'")["client_id"].nunique()
cli_uk = cmd.query("pays == 'Royaume-Uni'")["client_id"].nunique()

print(cli_irl, "clients irlandais |", cli_uk, "clients britanniques")

In [ ]:
verifier("7a - clients irlandais", cli_irl == 2, "nunique() et non count()")
verifier("7b - clients britanniques", cli_uk == 235, "nunique() sur client_id")

### Exercice 8 — La taille de l'effet

> **Votre mission :**
> - Calculer l'écart en euros entre le panier irlandais et le britannique → `ecart_eur` (arrondi à 2 décimales).
> - Et le rapport entre les deux → `rapport` (arrondi à 2 décimales).
> - Un test dit si un écart existe ; ces deux nombres disent s'il compte.

In [ ]:
irl = cmd.query("pays == 'Irlande'")["ca"]
uk = cmd.query("pays == 'Royaume-Uni'")["ca"]

ecart_eur = round(irl.mean() - uk.mean(), 2)
rapport = round(irl.mean() / uk.mean(), 2)

print(ecart_eur, "euros d'ecart |", rapport, "fois plus")

# A mettre systematiquement A COTE de la p-value : elle dit qu'un ecart
# existe, ces deux nombres disent s'il vaut une decision.

In [ ]:
verifier("8a - ecart en euros", ecart_eur == 626.46, "moyenne irlandaise moins britannique")
verifier("8b - rapport", rapport == 2.59, "divisez la moyenne irlandaise par la britannique")

### Exercice 9 — Vingt tests sur rien

> **Votre mission :**
> - Refaire les 20 comparaisons entre deux moitiés du fichier tirées au hasard.
> - Compter combien de p-values passent sous 0,05 → `nb_sig`.
> - Rappel : il n'y a **aucune** différence à trouver, les groupes sont tirés au hasard.

In [ ]:
ps = []
for i in range(20):
    a = cmd.sample(frac=0.5, random_state=i)
    b = cmd.drop(a.index)
    ps.append(stats.ttest_ind(a["ca"], b["ca"], equal_var=False).pvalue)

nb_sig = (pd.Series(ps) < 0.05).sum()
print(nb_sig, "test(s) significatif(s) sur 20, alors qu'il n'y a rien a trouver")

# Un sur vingt, soit exactement 5 % : c'est la definition du seuil,
# pas un accident. Tester tout contre tout garantit de "trouver".

In [ ]:
verifier("9 - faux positifs sur 20 tests", nb_sig == 1,
         "le seuil de significativite usuel est 0,05")

### Exercice 10 — Question de synthèse

> **Votre mission :**
> - On vous demande une recommandation écrite sur l'arbitrage France / Allemagne.
> - Calculer l'écart observé entre les deux moyennes → `ecart_fr_de` (arrondi à 2 décimales).
> - Récupérer la p-value du test → `p_fr_de` (arrondie à 3 décimales).
> - Puis rédigez votre recommandation en commentaire, en trois phrases maximum.

In [ ]:
ecart_fr_de = round(de.mean() - fr.mean(), 2)
p_fr_de = round(stats.ttest_ind(fr, de, equal_var=False).pvalue, 3)

print("ecart :", ecart_fr_de, "euros | p =", p_fr_de)

# Recommandation possible :
# "L'ecart de panier moyen entre l'Allemagne et la France est de 10,71 EUR,
#  soit 2 % — et il n'est pas mesurable sur nos volumes actuels (p = 0,87).
#  Ces donnees ne permettent pas d'arbitrer entre les deux marches : il faut
#  trancher sur le cout d'acquisition ou la marge, pas sur le panier."

In [ ]:
verifier("10a - ecart France/Allemagne", ecart_fr_de == 10.71,
         "moyenne allemande moins moyenne francaise")
verifier("10b - p-value", p_fr_de == 0.874, "arrondissez a 3 decimales")